# Relative GCN Prediction Processing

This notebook processes relative GCN predictions and converts them to absolute predictions for calixarene host molecules.

## Overview

The workflow involves:
1. Loading and preprocessing SMILES data for calixarene molecules
2. Processing relative GCN predictions from pickled model outputs
3. Converting relative predictions to absolute predictions using reference molecules
4. Analyzing prediction accuracy with different approaches (with/without reversed pairs)
5. Calculating final results and model performance metrics

## Input Data

- CSV file containing calixarene SMILES and target binding values
- Pickled file with relative GCN predictions between molecule pairs

## Output

- Processed results dictionary with absolute predictions for each host molecule
- Model performance metrics including R² scores


## Setup and Dependencies


In [ ]:
# Dependencies: tensorboardX, rdkit, rdkit-pypi
# Install via: pip install tensorboardX rdkit rdkit-pypi

## Data Processing Functions


In [4]:
import pickle
import pandas as pd
from rdkit import Chem
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:

def prepare_data(raw_filename, target_name='Calx'):
    """
    Process SMILES data from CSV file and convert to canonical form.
    
    Parameters:
    -----------
    raw_filename : str
        Path to the CSV file containing SMILES data
    target_name : str, default='Calx'
        Target column name (currently unused)
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with processed canonical SMILES and original data
    """
    smiles_targets_df = pd.read_csv(raw_filename)
    print("Number of all SMILES:", len(smiles_targets_df))

    remained_smiles = []
    canonical_smiles_list = []
    atom_num_dist = []
    for smiles in smiles_targets_df['SMILES']:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                atom_num_dist.append(len(mol.GetAtoms()))
                remained_smiles.append(smiles)
                canonical_smiles = Chem.MolToSmiles(mol, isomericSmiles=True)
                canonical_smiles_list.append(canonical_smiles)
        except:
            continue

    remained_df = pd.DataFrame({
        'SMILES': remained_smiles,
        'cano_smiles': canonical_smiles_list
    })
    for col in smiles_targets_df.columns:
        if col != 'SMILES':
            remained_df[col] = smiles_targets_df[smiles_targets_df['SMILES'].isin(remained_smiles)][col].values

    return remained_df


remained_df = prepare_data('/notebooks/Codebase/Database/cal_abs.csv')



Number of all SMILES: 38


## Load and Process Data


In [ ]:
def process_dictionary(df, unprocessed_dict, relative_is_counter_minus_host=True):
    """
    Process the unprocessed dictionary to calculate absolute predictions for each host.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame containing host information with columns 'Host', 'cano_smile', and target columns
    unprocessed_dict : dict
        Dictionary with hosts as main keys, smile pairs as subkeys, and target predictions
    relative_is_counter_minus_host : bool, default=True
        If True, relative prediction is (counter - host)
        If False, relative prediction is (host - counter)
    
    Returns:
    --------
    dict
        Processed dictionary with average absolute predictions for each host and target
    """
    processed_dict = {}
    
    for host, smile_pairs in unprocessed_dict.items():
        host_smile = df.loc[df['Host'] == host, 'cano_smile'].values[0]
        processed_dict[host] = {}
        
        for smile_pair, targets in smile_pairs.items():
            smile1, smile2 = smile_pair.split(',')
            
            if smile1.strip() == host_smile:
                counter_smile = smile2.strip()
            elif smile2.strip() == host_smile:
                counter_smile = smile1.strip()
            else:
                continue
            
            counter_row = df[df['cano_smile'] == counter_smile]
            if counter_row.empty:
                continue
            
            for target_name, values in targets.items():
                if target_name not in processed_dict[host]:
                    processed_dict[host][target_name] = {'predictions': []}
                
                counter_actual = counter_row[target_name].values[0]
                relative_prediction = values['predicted']
                
                if relative_is_counter_minus_host:
                    host_absolute_prediction = counter_actual - relative_prediction
                else:
                    host_absolute_prediction = relative_prediction + counter_actual
                
                processed_dict[host][target_name]['predictions'].append(host_absolute_prediction)
        
        for target_name in list(processed_dict[host].keys()):
            predictions = processed_dict[host][target_name]['predictions']
            if predictions:
                processed_dict[host][target_name] = {
                    'actual': host_row[target_name],
                    'predicted': sum(predictions) / len(predictions)
                }
            else:
                del processed_dict[host][target_name]
    
    return processed_dict
unprocessed_df = pd.read_pickle('/notebooks/Codebase/GCN/CheckPoint/RGCN_regression_unprocessed.pkl')
data = unprocessed_df


NameError: name 'pd' is not defined

## Prediction Analysis


In [ ]:
def compare_predictions_with_without_reversed(nested_dict, remained_df):
    """
    Compare predictions with and without handling reversed pairs.
    
    Parameters:
    -----------
    nested_dict : dict
        Dictionary containing prediction data
    remained_df : pandas.DataFrame
        DataFrame with canonical SMILES and target values
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with comparison results
    """
    abs_values_lookup = remained_df.set_index('cano_smiles').to_dict(orient='index')
    comparison_results = []
    
    for main_smiles, pairs in nested_dict.items():
        for pair, targets in pairs.items():
            smiles1, smiles2 = pair.split(',')
            
            if main_smiles == smiles1:
                other_smiles = smiles2
                straight = True
            else:
                other_smiles = smiles1
                straight = False
            
            if other_smiles in abs_values_lookup:
                abs_values_other = abs_values_lookup[other_smiles]
                
                for target, values in targets.items():
                    rel_predicted = values['predicted']
                    abs_other_value = abs_values_other[target]
                    
                    if straight:
                        predicted_main_with_reversed = rel_predicted + abs_other_value
                    else:
                        predicted_main_with_reversed = abs_other_value - rel_predicted
                    
                    predicted_main_without_reversed = rel_predicted + abs_other_value
                    actual_value = abs_values_lookup[main_smiles][target]
                    
                    comparison_results.append({
                        'main_smiles': main_smiles,
                        'pair': pair,
                        'target': target,
                        'predicted_with_reversed': predicted_main_with_reversed,
                        'predicted_without_reversed': predicted_main_without_reversed,
                        'actual': actual_value,
                        'error_with_reversed': abs(predicted_main_with_reversed - actual_value),
                        'error_without_reversed': abs(predicted_main_without_reversed - actual_value)
                    })
    
    return pd.DataFrame(comparison_results)

comparison_df = compare_predictions_with_without_reversed(data, remained_df)
mae_with_reversed = comparison_df['error_with_reversed'].mean()
mae_without_reversed = comparison_df['error_without_reversed'].mean()

print(f"MAE with reversed: {mae_with_reversed}")
print(f"MAE without reversed: {mae_without_reversed}")


                                             main_smiles  \
0      O=C(O)c1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc...   
1      O=C(O)c1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc...   
2      O=C(O)c1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc...   
3      O=C(O)c1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc...   
4      O=C(O)c1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc...   
...                                                  ...   
24955  O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...   
24956  O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...   
24957  O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...   
24958  O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...   
24959  O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1...   

                                                    pair    target  \
0      O=C(N[C@@H](Cc1ccc(O)cc1)C(=O)O)c1ccc(-c2cc3c(...      H3K4   
1      O=C(N[C@@H](Cc1ccc(O)cc1)C(=O)O)c1ccc(-c2cc3c(...    H3K4ac   
2      O=C(N[C@@H](Cc1ccc(O)cc1)C(=O)O)c1ccc(-c2cc3c(...   H3K4me1   

In [ ]:
def calculate_absolute_predictions(comparison_df):
    """
    Calculate final absolute predictions for each main SMILES.
    
    Parameters:
    -----------
    comparison_df : pandas.DataFrame
        DataFrame with comparison results
    
    Returns:
    --------
    tuple
        Two DataFrames with absolute predictions (with and without reversed handling)
    """
    abs_predictions_with_reversed = comparison_df.groupby(['main_smiles', 'target'])['predicted_with_reversed'].mean().reset_index()
    abs_predictions_without_reversed = comparison_df.groupby(['main_smiles', 'target'])['predicted_without_reversed'].mean().reset_index()
    
    actual_values = comparison_df.groupby(['main_smiles', 'target'])['actual'].mean().reset_index()
    
    abs_predictions_with_reversed = pd.merge(abs_predictions_with_reversed, actual_values, on=['main_smiles', 'target'])
    abs_predictions_without_reversed = pd.merge(abs_predictions_without_reversed, actual_values, on=['main_smiles', 'target'])
    
    return abs_predictions_with_reversed, abs_predictions_without_reversed

abs_predictions_with_reversed, abs_predictions_without_reversed = calculate_absolute_predictions(comparison_df)

mae_abs_with_reversed = (abs(abs_predictions_with_reversed['predicted_with_reversed'] - abs_predictions_with_reversed['actual'])).mean()
mae_abs_without_reversed = (abs(abs_predictions_without_reversed['predicted_without_reversed'] - abs_predictions_without_reversed['actual'])).mean()

print(f"MAE of absolute predictions (with reversed): {mae_abs_with_reversed}")
print(f"MAE of absolute predictions (without reversed): {mae_abs_without_reversed}")


MAE of absolute predictions (with reversed): 1.4373425649402456
MAE of absolute predictions (without reversed): 1.4196253464803643


In [ ]:
abs_predictions_with_reversed = abs_predictions_with_reversed.copy()
abs_predictions_without_reversed = abs_predictions_without_reversed.copy()

abs_predictions_with_reversed['error_with_reversed'] = abs(abs_predictions_with_reversed['predicted_with_reversed'] - abs_predictions_with_reversed['actual'])
abs_predictions_without_reversed['error_without_reversed'] = abs(abs_predictions_without_reversed['predicted_without_reversed'] - abs_predictions_without_reversed['actual'])

comparison_abs_errors = pd.merge(
    abs_predictions_with_reversed[['main_smiles', 'target', 'error_with_reversed']],
    abs_predictions_without_reversed[['main_smiles', 'target', 'error_without_reversed']],
    on=['main_smiles', 'target']
)

outliers_abs_errors = comparison_abs_errors[comparison_abs_errors['error_without_reversed'] < comparison_abs_errors['error_with_reversed']]

smiles_to_host = remained_df.set_index('cano_smiles')['Host'].to_dict()
outliers_abs_errors = outliers_abs_errors.copy()
outliers_abs_errors['Host'] = outliers_abs_errors['main_smiles'].map(smiles_to_host)

outliers_abs_errors_cleaned = outliers_abs_errors.dropna(subset=['Host'])
outliers_abs_errors_cleaned['Host'].unique()


/tmp/ipykernel_482/2655901289.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outliers_abs_errors['Host'] = outliers_abs_errors['main_smiles'].map(smiles_to_host)


array(['BM1', 'DM1', 'AM1', 'AP3', 'DO2', 'AO2', 'AP4', 'E8', 'AO1',
       'AP7', 'E6', 'E7', 'E3', 'E5', 'CP1', 'BP1', 'AP1', 'CP2', 'DP2',
       'E1', 'AM2', 'BP0', 'BH2', 'DO3', 'AH4', 'AH1', 'AP5', 'AO3',
       'AH2', 'AH3', 'AH7', 'PSC4', 'PSC6', 'P-NO2'], dtype=object)

In [ ]:
def calculate_predicted_main_values(nested_dict, remained_df):
    """
    Calculate predicted main values from relative predictions.
    
    Parameters:
    -----------
    nested_dict : dict
        Dictionary containing relative prediction data
    remained_df : pandas.DataFrame
        DataFrame with canonical SMILES and target values
    
    Returns:
    --------
    dict
        Dictionary with predicted values for each main SMILES and target
    """
    results = {}
    abs_values_lookup = remained_df.set_index('cano_smiles').to_dict(orient='index')
    
    for main_smiles, pairs in nested_dict.items():
        predicted_sums = {}
        counts = {}
        
        for pair, targets in pairs.items():
            smiles1, smiles2 = pair.split(',')
            
            if main_smiles == smiles1:
                other_smiles = smiles2
                straight = True
            else:
                other_smiles = smiles1
                straight = False
            
            if other_smiles in abs_values_lookup:
                abs_values_other = abs_values_lookup[other_smiles]
                
                for target, values in targets.items():
                    rel_predicted = values['predicted']
                    abs_other_value = abs_values_other[target]
                    
                    if straight:
                        predicted_main_value = rel_predicted + abs_other_value
                    else:
                        predicted_main_value = abs_other_value - rel_predicted
                    
                    if target not in predicted_sums:
                        predicted_sums[target] = 0
                        counts[target] = 0
                    
                    predicted_sums[target] += predicted_main_value
                    counts[target] += 1
        
        results[main_smiles] = {target: predicted_sums[target] / counts[target] for target in predicted_sums}
    
    return results

results = calculate_predicted_main_values(data, remained_df)

{'O=C(O)c1ccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc(c2O)Cc2cc(S(=O)(=O)O)cc(c2O)Cc2cc(-c4ccc(C(=O)O)cc4)cc(c2O)C3)cc1': {'H3K4': 3.2458737845540018, 'H3K4ac': 4.423415952373506, 'H3K4me1': 0.27263933384928074, 'H3K4me2': -1.343631476190112, 'H3K4me3': -1.8126681738973007, 'H3K9me3': -0.9887242198671214, 'H3R2me2a': -0.5269704717340362, 'H3R2me2s': 0.9555513358626624}, 'O=C(Nc1cc2c(O)c(c1)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)C2)c1ccc(-c2ccccc2)cc1': {'H3K4': -0.49539432668278965, 'H3K4ac': 0.4692139970273224, 'H3K4me1': -2.3904447561809166, 'H3K4me2': -4.500138081674006, 'H3K4me3': -3.4425524908960674, 'H3K9me3': -1.2954264892493346, 'H3R2me2a': -1.6185198114543238, 'H3R2me2s': -1.723560275068208}, 'CC(=O)c1cccc(-c2cc3c(O)c(c2)Cc2cc(S(=O)(=O)O)cc(c2O)Cc2cc(-c4cccc(C(C)=O)c4)cc(c2O)Cc2cc(S(=O)(=O)O)cc(c2O)C3)c1': {'H3K4': -1.1453067143335622, 'H3K4ac': -0.28793269556861695, 'H3K4me1': -0.7730002694756226, 'H3K4me2': -2.089378650362726, 'H3K4me3': -2.431861

In [ ]:
def process_results_with_host_and_actuals(results, remained_df):
    """
    Process results by mapping SMILES to host names and adding actual values.
    
    Parameters:
    -----------
    results : dict
        Dictionary with predicted values for each SMILES and target
    remained_df : pandas.DataFrame
        DataFrame with canonical SMILES, host names, and actual values
    
    Returns:
    --------
    dict
        Dictionary with processed results using host names as keys
    """
    smiles_to_host = dict(zip(remained_df['cano_smiles'], remained_df['Host']))
    processed_results = {}

    for main_smiles, target_dict in results.items():
        host = smiles_to_host.get(main_smiles, main_smiles)
        processed_results[host] = {}

        for target, predicted_value in target_dict.items():
            try:
                actual_value = remained_df.loc[remained_df['cano_smiles'] == main_smiles, target].values[0]
            except IndexError:
                print(f"Warning: Actual value for {main_smiles} and {target} not found")
                actual_value = None

            processed_results[host][target] = {
                'predicted': predicted_value,
                'actual': actual_value
            }

    return processed_results

processed_results = process_results_with_host_and_actuals(results, remained_df)


{'DP2': {'H3K4': {'predicted': 3.2458737845540018, 'actual': -0.356674944},
  'H3K4ac': {'predicted': 4.423415952373506, 'actual': 0.09531018},
  'H3K4me1': {'predicted': 0.27263933384928074, 'actual': -1.386294361},
  'H3K4me2': {'predicted': -1.343631476190112, 'actual': -2.302585093},
  'H3K4me3': {'predicted': -1.8126681738973007, 'actual': -3.244193633},
  'H3K9me3': {'predicted': -0.9887242198671214, 'actual': -2.645075402},
  'H3R2me2a': {'predicted': -0.5269704717340362, 'actual': -2.430418465},
  'H3R2me2s': {'predicted': 0.9555513358626624, 'actual': -1.347073648}},
 'E3': {'H3K4': {'predicted': -0.49539432668278965, 'actual': 0.09531018},
  'H3K4ac': {'predicted': 0.4692139970273224, 'actual': 0.993251773},
  'H3K4me1': {'predicted': -2.3904447561809166, 'actual': -1.049822124},
  'H3K4me2': {'predicted': -4.500138081674006, 'actual': -2.995732274},
  'H3K4me3': {'predicted': -3.4425524908960674, 'actual': -2.488914671},
  'H3K9me3': {'predicted': -1.2954264892493346, 'actua

## Calculate Final Predictions


In [9]:
with open('/notebooks/Result_dicts/Relative_FP.pkl', 'wb') as f:
    pickle.dump(processed_results, f)



In [ ]:



from sklearn.metrics import r2_score

def calculate_r2_from_processed_results(processed_results):
    """
    Calculate R² score from processed results.
    
    Parameters:
    -----------
    processed_results : dict
        Dictionary with processed results containing actual and predicted values
    
    Returns:
    --------
    float
        R² score
    """
    all_actual = []
    all_predicted = []
    
    for host, targets in processed_results.items():
        for target, values in targets.items():
            if isinstance(values['actual'], (int, float)) and values['actual'] is not None:
                all_actual.append(values['actual'])
                all_predicted.append(values['predicted'])
    
    if all_actual and all_predicted:
        return r2_score(all_actual, all_predicted)
    else:
        return None

overall_r2 = calculate_r2_from_processed_results(processed_results)
if overall_r2 is not None:
    print(f"Overall R² score: {overall_r2}")
else:
    print("No numeric data available for R² calculation")


Processing main SMILES: O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(-c3cccnc3)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(-c3cccnc3)cc(c1O)C2
Corresponding Host: BH2
Comparison of Average Predictions and Actual Values:
SMILES: Skipped (non-numeric value)
Host: Skipped (non-numeric value)
H3K4: Predicted = -2.023741404553652, Actual = 4.605170186
H3K4ac: Predicted = -1.6145652482054869, Actual = 0.693147181
H3K4me1: Predicted = -1.8221234277159346, Actual = 1.386294361
H3K4me2: Predicted = -2.9092618644183763, Actual = -0.385662481
H3K4me3: Predicted = -3.2568873104845406, Actual = 0.741937345
H3K9me3: Predicted = -2.037108704082214, Actual = 1.386294361
H3R2me2a: Predicted = -1.667801692581147, Actual = 0.587786665
H3R2me2s: Predicted = -2.1979364098807, Actual = 1.410986974
--------------------------------------------------
Processing main SMILES: O=S(=O)(O)c1cc2c(O)c(c1)Cc1cc(-c3ccccc3)cc(c1O)Cc1cc(S(=O)(=O)O)cc(c1O)Cc1cc(-c3ccccc3)cc(c1O)C2
Corresponding Host: BP0
Comparison of Average Predictions and A